In [ ]:
#! pip install -r requirements.txt

In [ ]:
#add par to source

import sys

sys.path.append('..')

## Components

### Data

#### Open Data

#### Risk Free

In [2]:
from tradepilot.data import OpenData

dataapi = OpenData()

dataapi.get_risk_free_rate(
    start='2020-01-01',
    end='2020-12-31',
)

Date
2020-01-02 00:00:00-06:00    1.495
2020-01-03 00:00:00-06:00    1.473
2020-01-06 00:00:00-06:00    1.488
2020-01-07 00:00:00-06:00    1.500
2020-01-08 00:00:00-06:00    1.493
                             ...  
2020-12-23 00:00:00-06:00    0.075
2020-12-24 00:00:00-06:00    0.075
2020-12-28 00:00:00-06:00    0.080
2020-12-29 00:00:00-06:00    0.090
2020-12-30 00:00:00-06:00    0.070
Name: Close, Length: 252, dtype: float64

#### Stock Market

##### live price

In [ ]:
from tradepilot.data import MarketData

dataapi = MarketData()
# live data
dataapi.get_live_price("AAPL")

In [ ]:
from tradepilot.data import MarketData

dataapi = MarketData()
# live data
dataapi.get_live_price(["AAPL", "MSFT", "GOOGL"])

##### Historic Price Data

In [ ]:
from tradepilot.data import MarketData

dataapi = MarketData()
dataapi.get_historical_data("AAPL", "2025-01-01", "2025-01-10")

In [ ]:
# multiple tickers
from tradepilot.data import MarketData

dataapi = MarketData()
dataapi.get_historical_data(["AAPL", "MSFT"], "2025-01-01", "2025-01-10")


### Metrics

#### Momentum

In [ ]:
from tradepilot.metrics import momentum
from tradepilot.data import MarketData
dataapi = MarketData()
prices = dataapi.get_historical_data("AAPL", "2025-01-01", "2025-01-10")
# periods to calculate momentum
# 5 days
momentum(prices, 5)

#### get_returns

In [ ]:
from tradepilot.metrics import get_returns
from tradepilot.data import MarketData

dataapi = MarketData()
prices = dataapi.get_historical_data("AAPL", "2025-01-01", "2025-01-10")
get_returns(prices)


In [ ]:
from tradepilot.data import OpenData

dataapi = OpenData()

dataapi.get_risk_free_rate(
    start='2020-01-01',
    end='2020-12-31',
)

#### Annualized Returns

#### Sharpe Ratio

#### max_drawdown

### Ranking

#### Latest Momentum

In [ ]:
from tradepilot.ranking import momentum_ranking

dataapi = MarketData()
timeseries = dataapi.get_historical_data(["AAPL", "TSLA"], "2025-01-01", "2025-01-10")

# 3 periods momentum
t = 3
momentum_ranking(timeseries, t)

#### Random Ranking

In [ ]:
from tradepilot.ranking import random_ranking

dataapi = MarketData()
timeseries = dataapi.get_historical_data("AAPL", "2025-01-01", "2025-01-10")

seed = 42
random_ranking(timeseries, seed)

## Simulation

In [1]:
# TPS class

from tradepilot.simulator import TPS
from tradepilot.data import MarketData, OpenData
from strategies.momentum import momentum_strategy
dataapi = MarketData()
opendataapi = OpenData()

data_start = "2024-01-01"
start = "2025-01-01"
end = "2025-01-14"

# get historical data
data = dataapi.get_historical_data(["AAPL", "TSLA"], data_start, end)
risk_free = opendataapi.get_risk_free_rate(
    start=data_start,
    end=end,
)
# create TPS object
tps = TPS(
    universe=data,
    criteria=momentum_strategy,
    #strategy_args={"t": 3},
    initial_capital=1_000_000,
    #rebalance_freq="W-MON",
    risk_free=risk_free,
    start=start,
    end=end,
    t=3,
    window=5
    )
    
res = tps.run(
    #start="2025-01-06",
    #end="2025-01-14"
    )


The first trading day is 2025-01-06


Simulating timelapse:   0%|          | 0/2 [00:00<?, ?it/s]/Users/artemiopadilla/Documents/repos/GitHub/personal/TradePilot/tradepilot/simulator.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stock_prices_window.dropna(inplace=True, axis=1)
Simulating timelapse:  50%|█████     | 1/2 [00:00<00:00, 157.38it/s]

Top stocks for 2025-01-06 00:00:00:
Index(['TSLA', 'AAPL'], dtype='object')


AttributeError: 'Series' object has no attribute 'append'

In [ ]:
tps.portfolio_weights

In [ ]:
tps.portfolio_values

In [ ]:
# remove data Date index timezone
data.index = data.index.tz_localize(None)
data.head()

In [ ]:
tps.universe.index

In [ ]:
# examples/example_backtest.py
from tradepilot.backtest import Backtest
from tradepilot.data import MarketData
from strategies.momentum import momentum_strategy

# Retrieve historical data for an asset (e.g., AAPL)
market_data = MarketData()
# For demonstration, we use a single asset; in real applications, use a DataFrame of multiple assets.
universe = market_data.get_historical_data("AAPL", "2020-01-01", "2025-01-31")

# Set up the backtest with the momentum strategy
backtest = Backtest(universe, momentum_strategy, initial_capital=1_000_000, risk_free=0.02)
backtest.run(start="2025-01-06", end="2025-01-14")
results = backtest.evaluate()

print("Backtest Metrics:")
for key, value in results.items():
    print(f"{key}: {value}")


# Trading

In [ ]:
# examples/example_live_trading.py
from tradepilot.trader import TPT
from tradepilot.broker import BrokerAPI
from tradepilot.data import MarketData
from strategies.momentum import momentum_strategy

# Retrieve recent data for an asset (e.g., AAPL)
market_data = MarketData()
universe = market_data.get_historical_data("AAPL", "2023-01-01", "2024-01-01")

# Set up and execute live trading (ensure you have proper API credentials and a paper trading account)
trader = TPT(broker_api="alpaca", universe=universe, strategy=momentum_strategy,
              capital=10000, risk_free=0.02)
trader.run()
